# R1000 Top30 Institutional Engine

GitHub `master` 기준 Colab 실행 노트북입니다.

실행 순서:
1. 의존성 설치
2. GitHub `master` 코드 가져오기 + Drive 출력 폴더 연결
3. 첫 풀런용 collector 실행
4. 본 파이프라인 실행 + validation
5. 결과 파일 직접 확인


In [ ]:
%pip install -q --upgrade pip
%pip install -q catboost pandas-market-calendars yfinance pyarrow openpyxl requests scikit-learn

import os
os.environ["ALPHA_VANTAGE_API_KEY"] = "JOUOW3UV8ZV23AOZ"

print("deps + env ready")


In [ ]:
# 1) GitHub master 코드 가져오기 + Drive 출력 폴더 연결
from google.colab import drive
import os
import sys
import json
import importlib
import subprocess
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

drive.mount('/content/drive', force_remount=False)

BASE_DIR = '/content/drive/MyDrive/r1000_top30_institutional'
DATA_DIR = Path(BASE_DIR)
REPO_DIR = Path('/content/r1000-quant-engine')
REPO_URL = 'https://github.com/wscha231/r1000-quant-engine.git'
BRANCH = 'master'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)

for name in ['r1000_top30_institutional', 'r1000_data_collector']:
    if name in sys.modules:
        del sys.modules[name]
importlib.invalidate_caches()

END_DATE = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d')

print('리포지토리 경로:', os.getcwd())
print('리포지토리 py 파일:', [x.name for x in REPO_DIR.glob('*.py')])
print('Drive 데이터 경로:', str(DATA_DIR))
print('기준 종료일:', END_DATE)


In [ ]:
# 2) 첫 풀런용 collector 실행
# - SEC / FRED / FSDS / Yahoo quarterly 기반
# - Alpha Vantage 무료 호출은 0으로 제어해서 한도 오류를 피함
from r1000_data_collector import collector_lean_full_run_cfg, run_data_collection

cfg = collector_lean_full_run_cfg(BASE_DIR, end_date=END_DATE)
cfg['sec_user_agent'] = 'R1000InstitutionalBot (contact: andrewcha231@gmail.com)'
cfg['fred_api_key'] = '8d92fb5a5de226657d912fe0284dfc00'
cfg['macro_refresh_days'] = 0
cfg['live_refresh_days'] = 1
cfg['companyfacts_refresh_days'] = 7
cfg['alpha_vantage_free_refresh_tickers'] = 0
cfg['alpha_vantage_free_statement_repair_tickers'] = 0
cfg['alpha_vantage_free_statement_refresh_days'] = 7
cfg['macro_slow_release_lag_months'] = 1
cfg['cash_target_growth_cap'] = 0.03
cfg['cash_target_balanced_cap'] = 0.05
cfg['cash_target_mild_risk_cap'] = 0.10
cfg['core_compounder_sleeve_base_weight'] = 0.55
cfg['future_winner_sleeve_base_weight'] = 0.35
cfg['future_winner_sleeve_min_weight'] = 0.05
cfg['future_winner_sleeve_max_weight'] = 0.60
cfg['early_scout_sleeve_base_weight'] = 0.05
cfg['early_scout_sleeve_min_weight'] = 0.00
cfg['early_scout_sleeve_max_weight'] = 0.15

collector_summary = run_data_collection(cfg)
print('=== collector 산출 파일 ===')
print(collector_summary['output_files'])
print('\n=== collector 핵심 커버리지 ===')
print(collector_summary['core_latest_coverage'])


In [ ]:
# 3) 본 파이프라인 실행 + validation
# - walk-forward 학습
# - 최신 포트 생성
# - acceptance / sleeve / 리밸런싱 점검
from r1000_top30_institutional import run_default_pipeline
from r1000_data_collector import run_full_validation_suite

pipeline_cfg = dict(cfg)
pipeline_cfg['reuse_existing_artifacts'] = True
pipeline_cfg['resume_partial_walkforward'] = False
pipeline_cfg['reuse_phase4_models_for_latest_recommendations'] = False

result = run_default_pipeline(pipeline_cfg)
report = run_full_validation_suite(pipeline_cfg, rerun_pipeline=False)

print('=== acceptance_checks ===')
print(result['acceptance_checks'])
print('\n=== backtest_policy_snapshot ===')
print(report['backtest_policy_snapshot'])
print('\n=== rebalance_interval_comparison_snapshot ===')
print(report['rebalance_interval_comparison_snapshot'])
print('\n=== sleeve_policy_snapshot ===')
print(report['sleeve_policy_snapshot'])
print('\n=== ops_tracking_snapshot ===')
print(report['ops_tracking_snapshot'])
print('\n=== macro_scored_latest coverage ===')
print(report['coverage']['macro_scored_latest'])
print('\n=== portfolio_shape ===')
print(report['portfolio_shape'])


In [ ]:
# 4) 결과 파일 직접 확인
# - 슬리브 목표 비중 / 실제 비중
# - 최신 포트와 top30 점수
# - 리밸런싱 비교 결과
import pandas as pd

OUT = DATA_DIR / 'outputs'
REP = OUT / 'reports'

weights = json.loads((OUT / 'weights_latest.json').read_text(encoding='utf-8'))
run_summary = json.loads((OUT / 'run_summary.json').read_text(encoding='utf-8'))
portfolio = pd.read_csv(OUT / 'portfolio_latest.csv')
top30 = pd.read_csv(OUT / 'top30_latest.csv')
rebalance = pd.read_csv(REP / 'rebalance_interval_comparison.csv') if (REP / 'rebalance_interval_comparison.csv').exists() else pd.DataFrame()
sleeve_bt = pd.read_csv(REP / 'portfolio_sleeve_backtest_comparison.csv') if (REP / 'portfolio_sleeve_backtest_comparison.csv').exists() else pd.DataFrame()

print('weights_latest 슬리브 목표 비중:', weights.get('sleeve_target_weights'))
print('weights_latest 슬리브 실제 비중:', weights.get('sleeve_actual_weights'))
print('run_summary 슬리브 목표 비중:', run_summary.get('portfolio_sleeve_target_weights'))
print('run_summary 슬리브 실제 비중:', run_summary.get('portfolio_sleeve_actual_weights'))
print('run_summary 리밸런싱 액션:', run_summary.get('rebalance_action'))
print('run_summary 현재 리밸런싱 주기:', run_summary.get('active_rebalance_interval_months'))
print('champion 리밸런싱 정책:', run_summary.get('champion_rebalance_policy'))
print('actual/proxy 데이터 상태:', run_summary.get('actual_data_status'))
print('actual/proxy 데이터 커버리지:', run_summary.get('actual_data_coverage'))

display(portfolio[[
    'ticker',
    'weight',
    'portfolio_sleeve_label',
    'portfolio_sleeve_role',
    'portfolio_selection_path',
    'portfolio_sleeve_winner_engine',
    'portfolio_sleeve_engine_edge',
    'minervini_momentum_alive_score',
    'minervini_trend_template_score',
    'breakout_setup_quality_score',
    'broken_momentum_penalty',
    'fundamental_turnaround_acceleration_score',
    'fundamental_revenue_thrust_score',
    'fundamental_profit_cash_acceleration_score',
    'fundamental_turn_positive_score',
    'cashflow_inflection_under_loss_score',
    'portfolio_sleeve_confidence',
    'sleeve_target_core_compounder_weight',
    'sleeve_target_future_winner_weight',
    'sleeve_target_early_scout_weight',
    'future_winner_regime_strength',
    'early_scout_regime_strength',
    'rebalance_action',
    'active_rebalance_interval_months',
]].head(20))

display(top30[[
    'rank',
    'score_rank',
    'portfolio_rank',
    'ticker',
    'weight',
    'selected_for_portfolio',
    'selection_reason',
    'portfolio_selection_path',
    'portfolio_sleeve_role',
    'portfolio_sleeve_winner_engine',
    'portfolio_sleeve_engine_edge',
    'minervini_momentum_alive_score',
    'minervini_trend_template_score',
    'breakout_setup_quality_score',
    'broken_momentum_penalty',
    'fundamental_turnaround_acceleration_score',
    'fundamental_revenue_thrust_score',
    'fundamental_profit_cash_acceleration_score',
    'fundamental_turn_positive_score',
    'cashflow_inflection_under_loss_score',
    'score',
    'portfolio_sleeve_label',
    'portfolio_core_compounder_engine_score',
    'portfolio_future_winner_engine_score',
    'portfolio_early_scout_engine_score',
]].head(30))

display(sleeve_bt)

display(rebalance)


In [ ]:
scored = pd.read_csv(OUT / 'scored_latest.csv')

macro_cols = [
    'm2_yoy_lag1m',
    'fed_assets_bil',
    'reverse_repo_bil',
    'tga_bil',
    'net_liquidity_bil',
    'net_liquidity_change_1m_bil',
    'liquidity_impulse_score',
    'liquidity_drain_score',
]

display(scored[[c for c in macro_cols if c in scored.columns]].notna().mean().sort_values(ascending=False))
